# Python's `zip()` Function

**Sebastian Perdomo | IS 362 | Week 2**

`zip()` pairs up items from two or more lists so you can loop over them together.

Three things about it are easy to get wrong: what it gives back, what it does when
the lists are different lengths, and how to reverse it. This notebook covers all
three.

In [1]:
students = ["Tommy", "Fred", "Gail"]
scores = [88, 92, 79]

for name, score in zip(students, scores):
    print(name, score)

Tommy 88
Fred 92
Gail 79


## 1. `zip()` gives back an iterator, not a list

`zip()` does not return a list. It returns an iterator, which builds the pairs one
at a time as you read them.

An iterator can only be read once. After you read it, it is empty. Reading it a
second time gives you nothing and does not raise an error.

In [3]:
z = zip(students, scores)
print(type(z))

print("first pass: ", list(z))
print("second pass:", list(z))   # empty, and no error is raised

<class 'zip'>
first pass:  [('Tommy', 88), ('Fred', 92), ('Gail', 79)]
second pass: []


Wrap `zip()` in `list()` when you need the result more than once.

Reading one pair at a time means `zip()` never holds the whole result in memory,
so it works on large files. The trade-off is that you have to keep track of
whether the object has already been read.

## 2. Different lengths get cut off

When the lists are different lengths, `zip()` stops at the shortest one and drops
the rest. It does not raise an error.

If two columns come out of a source with different row counts, `zip()` returns a
result that is missing rows.

In [4]:
names = ["Tommy", "Fred", "Gail", "Sebastian", "Randy"]
emails = ["t@example.com", "f@example.com", "g@example.com"]

paired = list(zip(names, emails))

print("input rows: ", len(names))
print("output rows:", len(paired))
print(paired)

input rows:  5
output rows: 3
[('Tommy', 't@example.com'), ('Fred', 'f@example.com'), ('Gail', 'g@example.com')]


The two names are gone, and nothing in the output says anything was dropped.

This matters when you are pulling in data. You definitely don't want your scripts returning incorrect results.

## 3. `strict=True` raises an error instead

Python 3.10 added the `strict` setting. With `strict=True`, lists of different
lengths raise a `ValueError` instead of getting cut off.

Use it when the lists are supposed to match, so a mismatch shows up as an error
you can see.

In [5]:
import sys
print("Python version:", sys.version_info[:3])

try:
    list(zip(names, emails, strict=True))
except ValueError as e:
    print("ValueError:", e)

Python version: (3, 14, 6)
ValueError: zip() argument 2 is shorter than argument 1


## 4. `zip_longest()` keeps every row

Sometimes lists of different lengths are expected and you want to keep all the
rows. `itertools.zip_longest()` fills the gaps with a value you choose.

Which one to use:

| Goal | Use |
|---|---|
| Stop at the shortest, mismatch is expected | `zip()` |
| Mismatch means something went wrong | `zip(..., strict=True)` |
| Keep all rows and mark the gaps | `zip_longest(..., fillvalue=...)` |

In [6]:
from itertools import zip_longest

print(list(zip_longest(names, emails, fillvalue="MISSING")))

[('Tommy', 't@example.com'), ('Fred', 'f@example.com'), ('Gail', 'g@example.com'), ('Sebastian', 'MISSING'), ('Randy', 'MISSING')]


## 5. Unzipping with `zip(*pairs)`

`zip()` also reverses itself. The `*` splits a list of pairs into separate
arguments, which turns rows back into columns.

The results come back as tuples.

In [7]:
pairs = [("Tommy", 88), ("Fred", 92), ("Gail", 79)]

unzipped_names, unzipped_scores = zip(*pairs)

print(unzipped_names, type(unzipped_names))
print(unzipped_scores)

('Tommy', 'Fred', 'Gail') <class 'tuple'>
(88, 92, 79)


## 6. Building records from columns

The main use is building dictionaries. Pairing a header row with each data row
turns position-based data into named fields, which is the format most analysis
tools expect.

In [8]:
# Two parallel lists into a lookup dictionary
print(dict(zip(students, scores)))

# A header row plus data rows into labeled records
fields = ["name", "score", "grade"]
rows = [
    ("Tommy", 88, "B+"),
    ("Fred", 92, "A-"),
    ("Gail", 79, "C+"),
]

records = [dict(zip(fields, row)) for row in rows]

for record in records:
    print(record)

{'Tommy': 88, 'Fred': 92, 'Gail': 79}
{'name': 'Tommy', 'score': 88, 'grade': 'B+'}
{'name': 'Fred', 'score': 92, 'grade': 'A-'}
{'name': 'Gail', 'score': 79, 'grade': 'C+'}


## Takeaways

1. `zip()` returns an iterator you can only read once. Wrap it in `list()` to reuse it.
2. Lists of different lengths get cut off with no error, which can drop rows.
3. `strict=True` raises a `ValueError` when the lengths do not match.
4. `zip_longest()` keeps every row and fills the gaps.
5. `zip(*pairs)` turns rows back into columns and returns tuples.
6. `dict(zip(headers, row))` turns position-based data into named fields.